In [48]:
import pandas as pd

In [49]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torchvision.datasets import MNIST

In [50]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

trainset = MNIST(root="./data",train = True,download=True,transform=transform)
testset = MNIST(root="./data",train =False,download=True,transform=transform)

In [51]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

In [52]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,28,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(28,56,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(56,112,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*112,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)  
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)
        return x

In [53]:
model = CNN()

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [56]:
epochs = 10 

for epoch in range(epochs):
    epoch_training_loss = 0.0
    model.train()    
    for images, labels in trainloader:
        optimizer.zero_grad()      
        output = model(images) #FP
        loss = criterion(output,labels) #loss fnx
        loss.backward() #BP
        optimizer.step() #update params
        epoch_training_loss += loss.item()
    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 0.006371449377058834
epoch = 2/10 & loss = 0.00897281172949743
epoch = 3/10 & loss = 0.00752761985450891
epoch = 4/10 & loss = 0.005576099688262908
epoch = 5/10 & loss = 0.005894120911067016
epoch = 6/10 & loss = 0.007012014496798631
epoch = 7/10 & loss = 0.005464334310788641
epoch = 8/10 & loss = 0.0059032441193487915
epoch = 9/10 & loss = 0.0046393217473626085
epoch = 10/10 & loss = 0.0059994917332594965


In [59]:
#Evaluation our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy = {correct_labels/total_labels*100}")

accuracy = 99.25
